# VQA 최적화 솔루션

### 베이스라인 대비 주요 개선사항

| 항목 | 베이스라인 | 최적화 |
|------|-----------|--------|
| 학습 데이터 | 200개 | 5073 + dev 4413 (다수결) |
| Epochs | 1 | 3 |
| LoRA rank | r=8 | r=16 |
| 유효 배치 | 4 | 8 (batch=2 × accum=4) |
| LR 스케줄 | linear | cosine + warmup |
| Gradient Clipping | 없음 | max_norm=1.0 |
| 레이블 마스킹 | 없음 (전체 손실) | 답변 토큰만 손실 계산 |
| 추론 방식 | 텍스트 생성 | **logit 직접 비교** (a/b/c/d 확률) |

In [1]:
# 환경 확인
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

2.12.0.dev20260401+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [2]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 라이브러리 & 설정

In [ ]:
import os, math, random
from collections import Counter
from dataclasses import dataclass
from typing import Any

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ── 하이퍼파라미터 ──────────────────────────────────────────────────────────────
MODEL_ID      = "Qwen/Qwen2.5-VL-3B-Instruct"

IMAGE_SIZE    = 384
NUM_EPOCHS    = 3
BATCH_SIZE    = 1
GRAD_ACCUM    = 4          # 유효 배치 = BATCH_SIZE * GRAD_ACCUM = 4
LR            = 2e-4
MAX_GRAD_NORM = 1.0
LORA_R        = 16
LORA_ALPHA    = 32
WARMUP_RATIO  = 0.05
SEED          = 42
N_TRAIN       = 200        # 학습에 사용할 샘플 수
USE_DEV_DATA  = False
SAVE_DIR      = "./qwen_vqa_lora"

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

c:\Users\SSAFY\Desktop\AI\2026-ssafy-ai-15-2\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 2. 데이터 준비

- **train.csv**: 5073개 레이블 데이터
- **dev.csv**: 4413개, 5명 어노테이터 답변 → 다수결(2표 이상)로 레이블 생성

In [5]:
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# 150개 샘플만 사용
train_df = train_df.sample(n=N_TRAIN, random_state=SEED).reset_index(drop=True)

print(f"Train: {len(train_df)} 샘플")
print(f"Test:  {len(test_df)} 샘플")
train_df.head(3)

Train: 150 샘플
Test:  5074 샘플


,id,path,question,a,b,c,d,answer
0,train_1730.jpg,train/train_1730.jpg,사진 속 재활용 가능한 유리병의 개수는 몇 개인가요?,2개,1개,4개,3개,a
1,train_2655.jpg,train/train_2655.jpg,사진에 보이는 재활용 가능한 플라스틱 용기는 무엇인가요?,흰색 튜브형 용기,종이 냅킨,빨간색 마우스,꽃,a
2,train_0034.jpg,train/train_0034.jpg,사진에 보이는 재활용품 중 골판지 재질의 물건은 몇 개입니까?,2개,3개,1개,4개,b


## 3. 프롬프트 템플릿

In [6]:
SYSTEM_INSTRUCT = (
    "You are a visual question answering expert specializing in recyclable materials. "
    "Analyze the image carefully and answer with exactly one letter: a, b, c, or d. "
    "No explanation, no punctuation — just a single lowercase letter."
)

def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
    )

## 4. 모델 & Processor 로드

- 4-bit 양자화 (NF4)
- LoRA r=16 (베이스라인: r=8)

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
c:\Users\SSAFY\Desktop\AI\2026-ssafy-ai-15-2\baseline\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
W0402 10:39:42.234000 6096 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\SSAFY\Desktop\AI\2026-ssafy-ai-15-2\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning

trainable params: 37,152,768 || all params: 3,791,775,744 || trainable%: 0.9798


## 5. Dataset & DataCollator

### 레이블 마스킹 (핵심 개선)
- 베이스라인: 프롬프트 전체에 대해 손실 계산 → 비효율
- 최적화: **답변 토큰만** 손실 계산 (프롬프트 토큰은 labels=-100으로 마스킹)

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, train: bool = True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        user_text = build_mc_prompt(
            str(row["question"]), str(row["a"]), str(row["b"]),
            str(row["c"]), str(row["d"])
        )
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user",   "content": [{"type": "image", "image": img},
                                            {"type": "text",  "text": user_text}]},
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})
        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images, prompt_lens = [], [], []

        for sample in batch:
            messages = sample["messages"]
            img      = sample["image"]

            # 전체 텍스트 (어시스턴트 답변 포함)
            full_text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(full_text)
            images.append(img)

            if self.train:
                # 프롬프트만 인코딩해서 길이 측정 → 마스킹 경계
                prompt_text = self.processor.apply_chat_template(
                    messages[:-1], tokenize=False, add_generation_prompt=True
                )
                enc_p = self.processor(
                    text=[prompt_text], images=[img], return_tensors="pt"
                )
                prompt_lens.append(enc_p["input_ids"].shape[1])

        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            for i, plen in enumerate(prompt_lens):
                labels[i, :plen] = -100  # 프롬프트 토큰 손실 제외
            enc["labels"] = labels

        return enc

## 6. DataLoader

In [ ]:
train_df_shuffled = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split = int(len(train_df_shuffled) * 0.9)
train_subset = train_df_shuffled.iloc[:split]
valid_subset = train_df_shuffled.iloc[split:]

train_ds = VQAMCDataset(train_subset, train=True)
valid_ds = VQAMCDataset(valid_subset, train=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor, True), num_workers=0
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor, True), num_workers=0
)

print(f"학습: {len(train_subset)}개 / 검증: {len(valid_subset)}개")

## 7. Fine-tuning

- Cosine LR + Warmup (5%)
- Gradient Clipping (max_norm=1.0)
- 각 epoch 종료 후 best 모델 저장

In [ ]:
model = model.to(device)

num_update_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps = int(num_update_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_update_steps
)
scaler = torch.amp.GradScaler("cuda", enabled=True)

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    # ── 학습 ──
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = model(**batch).loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            avg = running_loss / GRAD_ACCUM
            pbar.set_postfix({"loss": f"{avg:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            running_loss = 0.0

    # ── 검증 ──
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]"):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    avg_val = val_loss / val_steps
    print(f"[Epoch {epoch+1}] val_loss: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  → Best model saved (val_loss={avg_val:.4f})")

print(f"\n학습 완료. Best val_loss: {best_val_loss:.4f}")

## 8. Logit 기반 추론 (핵심 개선)

### 왜 logit 기반 추론이 더 좋은가?
- **텍스트 생성 방식**: 모델이 토큰을 순차 생성 → `extract_choice()`로 파싱 → 실패 가능성
- **Logit 방식**: 어시스턴트 첫 토큰 위치에서 `a/b/c/d`의 확률만 직접 비교  
  → 항상 4개 중 하나 선택, 파싱 오류 없음, **약 2배 빠름**

In [ ]:
# a, b, c, d의 토큰 ID 확인
CHOICES = ["a", "b", "c", "d"]
choice_token_ids = []
for ch in CHOICES:
    ids = processor.tokenizer.encode(ch, add_special_tokens=False)
    choice_token_ids.append(ids[0])
    print(f"'{ch}' → token_id = {ids[0]}")
choice_token_ids = torch.tensor(choice_token_ids, device=device)


def predict_logit(img, question, a, b, c, d):
    """
    텍스트를 생성하지 않고 a/b/c/d 로짓을 직접 비교해 정답 반환.
    add_generation_prompt=True 로 인해 시퀀스 끝이 assistant 턴 시작부.
    logits[:, -1, :] = 다음 토큰(=정답) 예측 분포.
    """
    user_text = build_mc_prompt(question, a, b, c, d)
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user",   "content": [{"type": "image", "image": img},
                                        {"type": "text",  "text": user_text}]},
    ]
    prompt_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[prompt_text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        outputs = model(**inputs)

    last_logits    = outputs.logits[0, -1, :]         # (vocab_size,)
    choice_logits  = last_logits[choice_token_ids]     # (4,)
    return CHOICES[choice_logits.argmax().item()]

In [ ]:
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    pred = predict_logit(
        img, str(row["question"]),
        str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
    )
    preds.append(pred)

print("예측 분포:")
import pandas as pd
print(pd.Series(preds).value_counts().sort_index())

## 9. 제출 파일 생성

In [ ]:
import os
os.makedirs("content", exist_ok=True)

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("content/submission.csv", index=False)
print("Saved: content/submission.csv")
submission.head(10)